In [ ]:
import pandas as pd
import datetime as dt

# 1. Data-va load pandrom
df = pd.read_csv('global_ecommerce_sales.csv') # Corrected file name

# 2. Date column-a correct aana format-ku mathurom
df['Order_Date'] = pd.to_datetime(df['Order_Date'])

# 3. Kadasivya vithatha vachu oru 'Snapshot Date' set pandrom (RFM calculate panna)
snapshot_date = df['Order_Date'].max() + dt.timedelta(days=1)

# 4. RFM Logic - Ovvoru customer-kum calculate pandrom
rfm = df.groupby('Customer_Name').agg({
    'Order_Date': lambda x: (snapshot_date - x.max()).days,  # Recency: Evvalo naatkal aachu?
    'Order_ID': 'count',                                     # Frequency: Evvalo thadava vangirukkanga?
    'Total_Sales': 'sum'                                     # Monetary: Mothama evvalo selavu pannirukkanga?
}).reset_index()

# 5. Column perugalai theliva mathurom
rfm.rename(columns={
    'Order_Date': 'Recency',
    'Order_ID': 'Frequency',
    'Total_Sales': 'Monetary'
}, inplace=True)

# 6. Final result-a paaka
print(rfm.head())

    Customer_Name  Recency  Frequency  Monetary
0       Ahmed Ali      384          1    509.12
1  Ahmed Anderson      494          1    133.68
2     Ahmed Clark      636          2     94.28
3    Ahmed Dubois       23          3    105.73
4    Ahmed Garcia      923          2   1769.48


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 1. Data-va Standardize pandrom (Machine learning-ku data orey scale-la irukkanum)
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

# 2. K-Means AI Model-a create panni 4 Clusters-ah pirikirom
# n_clusters=4 na AI namma customers-a 4 groups-ah pirikkum
kmeans = KMeans(n_clusters=4, random_state=42)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

# 3. Ovvoru Cluster-kum peru vaikkurom (Business puriyumbadi)
cluster_names = {
    0: 'Lost Customers',      # Romba naal aachu, selavum pannala
    1: 'VIP / Champions',     # Adikkadi vanguvanga, adhigam selavu pannuvanga
    2: 'New / Low Spenders',  # Puthusa vanthavanga, kammiya vanguvanga
    3: 'At Risk'              # Nalla vangitu irunthavanga, ippo varala
}
# Note: K-Means cluster numbers random-ah aagum, final output vachu intha perugala namma Power BI-la adjust pannikalam.
# Ippo simple-ah AI kodutha number-aiye vechukalam.

# 4. Result-a print panni paakurom
print(rfm.head(10))

# 5. VERY IMPORTANT: Intha final data-va Power BI-ku anuppa CSV ah save pandrom!
rfm.to_csv('Final_RFM_Clusters.csv', index=False)
print("\nSuccess! Final data saved as 'Final_RFM_Clusters.csv'")

     Customer_Name  Recency  Frequency  Monetary  Cluster
0        Ahmed Ali      384          1    509.12        0
1   Ahmed Anderson      494          1    133.68        0
2      Ahmed Clark      636          2     94.28        2
3     Ahmed Dubois       23          3    105.73        2
4     Ahmed Garcia      923          2   1769.48        3
5     Ahmed Harris     1030          1    598.14        1
6     Ahmed Hassan      264          1    162.03        0
7    Ahmed Jackson      275          2    247.54        2
8  Ahmed Johansson      133          2   1670.10        3
9    Ahmed Johnson      383          1     27.50        0

Success! Final data saved as 'Final_RFM_Clusters.csv'
